# HOT trees dataset (`hotosm/fAIr-guru-td-trees`): label investigation

Purpose: understand how HOT's Freetown tree labels differ from the sources our training setup was built on, before the discussion with HOT about how to train on them.

Two differences stand out on the dataset card:

1. **Tree groups.** Dense or overlapping trees are mapped as `natural=tree_group` polygons. Our current training setup treats OAM-TCD canopy groups as *ignore* regions (masked out of training and evaluation). HOT may prefer the model to predict **one box per tree group**.
2. **Points, not crowns.** Isolated trees are mapped as **points** (`natural=tree`) with a mapper-estimated `diameter_crown`, not as crown polygons or boxes. A training box has to be derived from point + diameter.

Sections: 0 setup · 1 what the labels are · 2 points and `diameter_crown` · 3 tree groups · 4 what each labelling option gives the detector · 5 splits and leakage · 6 imagery with labels · 7 optional: release DeepForest model zero-shot · 8 summary and questions for HOT.

Runtime: CPU is enough for sections 0–6 (downloads about 125 MB, runs in a few minutes). Section 7 is off by default; it is faster on a GPU but runs on CPU (about 15 minutes for 12 blocks).

Labels come from the dataset's STAC files (`labels.parquet`, `tiles.parquet`: all 120k labels without imagery), so sections 1–5 cover all splits; imagery (sections 6–7) comes from one split.

In [1]:
REPO_ID = "hotosm/fAIr-guru-td-trees"
REVISION = "v0.1.0"  # the dataset version tag the card links to
IMAGE_SPLIT = "test"  # imagery for sections 6-7: "test" (116 MB) or "validation" (84 MB); train is 780 MB
UTM = "EPSG:32628"  # UTM 28N covers Freetown; used for metres
SEED = 0

# serving and anchor geometry of fair_trees (params.py, decode.py), used to judge box sizes
WINDOW_M = 40.0  # inference window side at the default window_overlap 0.5
WHOLE_OBJECT_M = WINDOW_M * 0.5  # window - stride: largest object always kept whole in one window
MODEL_PX = 800  # a window is resized to 800 px before the network
ANCHOR_MIN_PX, ANCHOR_MAX_PX = 32, int(512 * 2 ** (2 / 3))

In [2]:
import json
import re

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from scipy.spatial import cKDTree

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def fetch(path):
    return hf_hub_download(REPO_ID, path, repo_type="dataset", revision=REVISION)


labels_path = fetch("stac/project-50661/labels.parquet")  # every label, one row per (label, tile)
tiles_path = fetch("stac/project-50661/tiles.parquet")  # every tile footprint with its split
image_path = fetch(f"data/{IMAGE_SPLIT}-00000-of-00001.parquet")

labels = gpd.read_parquet(labels_path)
tiles = gpd.read_parquet(tiles_path)
tiles[["tile_x", "tile_y", "tile_z"]] = tiles.tile_id.str.extract(r"OAM-(\d+)-(\d+)-(\d+)").astype(int)
labels["tags"] = labels.tags.map(json.loads)
labels_m = labels.to_crs(UTM)
tiles_m = tiles.to_crs(UTM)
print(f"{len(labels):,} label rows, {labels.osm_id.nunique():,} OSM features, {len(tiles):,} tiles")
print("zoom:", tiles.tile_z.unique(), " tile side (m):", round(float(np.sqrt(tiles_m.area.median())), 2))
print("ground pixel size (m):", round(float(np.sqrt(tiles_m.area.median())) / 256, 4))

stac/project-50661/labels.parquet: reconstructing file:   0%|          |  0.00B / 6.57MB            

stac/project-50661/labels.parquet: downloading bytes:           |  0.00B            

stac/project-50661/tiles.parquet: reconstructing file:   0%|          |  0.00B / 1.40MB            

stac/project-50661/tiles.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  116MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

120,434 label rows, 106,632 OSM features, 48,490 tiles
zoom: [20]  tile side (m): 37.68
ground pixel size (m): 0.1472


## 1. What the labels are

One row per (label, tile): a polygon crossing a tile edge is clipped and appears once per tile it touches, so OSM feature counts and row counts differ.

In [3]:
GSD = float(np.sqrt(tiles_m.area.median())) / 256
labels["geom_type"] = labels.geom_type
print(pd.crosstab(labels["class"], labels.geom_type, margins=True), "\n")
print("rows vs unique OSM features per class:")
print(labels.groupby("class").agg(rows=("osm_id", "size"), features=("osm_id", "nunique")), "\n")
print("OSM element type:", labels.osm_type.value_counts().to_dict())
tag_keys = pd.Series([k for t in labels.tags for k in t]).value_counts()
print("tag keys:", tag_keys.head(10).to_dict())
leaf = labels.tags.map(lambda t: t.get("leaf_type")).fillna("(none)")
print("leaf_type on trees:", leaf[labels["class"] == "tree"].value_counts().to_dict())
print("last edit year:", pd.to_datetime(labels.osm_timestamp).dt.year.value_counts().sort_index().to_dict())

col_0       MultiPolygon  Point  Polygon     All
class                                           
tree                   0  95992        0   95992
tree_group          1165      2    23275   24442
All                 1165  95994    23275  120434 

rows vs unique OSM features per class:
             rows  features
class                      
tree        95992     95992
tree_group  24442     10640 

OSM element type: {'nodes': 95994, 'ways_poly': 24394, 'relations': 46}
tag keys: {'natural': 120434, 'diameter_crown': 92125, 'leaf_type': 18845, 'diameter': 2929, 'location': 52, 'type': 46, 'tree_type': 30, 'building': 21, 'construction': 19, 'species': 3}
leaf_type on trees: {'(none)': 77243, 'palm': 18731, 'broadleaved': 12, '3': 3, '4': 2, '7': 1}
last edit year: {2014: 36, 2023: 1, 2026: 120397}


## 2. Points and `diameter_crown`

A point gives a box only through `diameter_crown` (the box would be the point ± d/2). The card says the value is a mapper estimate in mixed formats. Below: how many are parseable, the distribution, values repeated suspiciously often (presets or copy-paste), and the resulting box size in pixels at z20 and in the model's 800 px window.

In [4]:
def parse_diameter(value):
    """Metres from strings like '3m', '2.412m', '3,134 m', '4'; NaN if missing or unclear."""
    if not isinstance(value, str):
        return np.nan
    m = re.fullmatch(r"\s*(\d+(?:[.,]\d+)?)\s*(?:m|M|metres|meters)?\s*", value)
    return float(m.group(1).replace(",", ".")) if m else np.nan


trees = labels_m[labels_m["class"] == "tree"].copy()
trees["d_raw"] = trees.tags.map(lambda t: t.get("diameter_crown"))
trees["d"] = trees.d_raw.map(parse_diameter)
raw = trees.d_raw.dropna()
print(f"trees: {len(trees):,}")
print(
    f"diameter_crown missing: {trees.d_raw.isna().mean():.1%}, unparseable: {(trees.d_raw.notna() & trees.d.isna()).sum()}"
)
print(
    f"with a unit: {raw.str.contains('[mM]').mean():.1%}, with >= 3 decimals: {raw.str.contains(r'[.,]\d{3}').mean():.1%}"
)
print("unparseable examples:", trees.d_raw[trees.d_raw.notna() & trees.d.isna()].unique()[:12].tolist())
print("\ndiameter (m) percentiles:")
print(trees.d.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2).to_string())
print("\nmost repeated values (m):", trees.d.value_counts().head(12).to_dict())
for lo, hi in [(0, 1), (1, 2), (20, 50), (50, np.inf)]:
    print(f"  d in ({lo}, {hi}] m: {((trees.d > lo) & (trees.d <= hi)).sum():,}")

d = trees.d.dropna()
print(
    f"\nbox side at z20: median {d.median() / GSD:.0f} px, p5 {d.quantile(0.05) / GSD:.0f} px, p95 {d.quantile(0.95) / GSD:.0f} px (256 px tile)"
)
px_800 = d / WINDOW_M * MODEL_PX
print(
    f"box side in the 800 px model input ({WINDOW_M:.0f} m window): median {px_800.median():.0f} px;"
    f" below the smallest anchor ({ANCHOR_MIN_PX} px = {ANCHOR_MIN_PX * WINDOW_M / MODEL_PX:.1f} m): {(px_800 < ANCHOR_MIN_PX).mean():.1%}"
)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
d.clip(upper=25).plot.hist(bins=100, ax=ax[0], title="diameter_crown (m), clipped at 25")
pal = trees.tags.map(lambda t: t.get("leaf_type") == "palm")
for name, sel in [("palm", pal), ("other", ~pal)]:
    trees.loc[sel, "d"].clip(upper=25).plot.hist(bins=50, ax=ax[1], alpha=0.6, density=True, label=name)
ax[1].set_title("diameter_crown by leaf_type")
ax[1].legend()
plt.show()

trees: 95,992
diameter_crown missing: 4.2%, unparseable: 41
with a unit: 59.8%, with >= 3 decimals: 25.5%
unparseable examples: ['-', 'é', "'", '(', "4m'", '3µ', '_', '4n', '5m#', '4,,03', 'qm', "5m'"]

diameter (m) percentiles:
count    91959.00
mean         4.58
std         83.61
min          1.00
1%           1.00
5%           2.00
25%          2.41
50%          4.00
75%          5.00
95%          8.24
99%         13.00
max      25231.00

most repeated values (m): {5.0: 15016, 2.0: 11089, 3.0: 10782, 4.0: 10731, 6.0: 4801, 1.0: 3795, 2.412: 3791, 2.231: 2927, 7.0: 2658, 8.0: 2122, 3.234: 1405, 10.0: 987}
  d in (0, 1] m: 3,795
  d in (1, 2] m: 11,360
  d in (20, 50] m: 230
  d in (50, inf] m: 6

box side at z20: median 27 px, p5 14 px, p95 56 px (256 px tile)
box side in the 800 px model input (40 m window): median 80 px; below the smallest anchor (32 px = 1.6 m): 4.4%


In [5]:
# spacing: how close are mapped trees, and would boxes built from point + diameter overlap?
xy = np.c_[trees.geometry.x, trees.geometry.y]
nn_dist, nn_idx = cKDTree(xy).query(xy, k=2)
trees["nn"] = nn_dist[:, 1]
print("nearest-neighbour distance (m):")
print(trees.nn.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75]).round(2).to_string())
both = trees.d.notna()
print(
    f"\ncrown discs overlapping their nearest neighbour (nn < d): {(trees.nn[both] < trees.d[both]).mean():.1%}"
)
print(f"near-duplicate points (< 0.5 m apart): {(trees.nn < 0.5).sum()}")

nearest-neighbour distance (m):
count    95992.00
mean        10.20
std          7.34
min          0.00
1%           1.94
5%           3.03
25%          5.48
50%          8.15
75%         12.64
max        224.41

crown discs overlapping their nearest neighbour (nn < d): 10.8%
near-duplicate points (< 0.5 m apart): 92


## 3. Tree groups

Each group is dissolved back from its per-tile pieces. Key questions for the "one box per group" option: how big groups are relative to a single crown and to the model's window; how much of a group's box is not group (fill ratio); how much area they cover.

In [6]:
groups_rows = labels_m[labels_m["class"] == "tree_group"]
groups = groups_rows.dissolve("osm_id").reset_index()
groups = groups[groups.area > 0]
groups["area_m2"] = groups.area
b = groups.bounds
groups["box_w"], groups["box_h"] = b.maxx - b.minx, b.maxy - b.miny
groups["box_long"] = groups[["box_w", "box_h"]].max(axis=1)
groups["fill"] = groups.area_m2 / (groups.box_w * groups.box_h)
crown_area = np.pi * (trees.d.median() / 2) ** 2
groups["crowns_equiv"] = groups.area_m2 / crown_area
groups["tiles"] = groups_rows.groupby("osm_id").size().reindex(groups.osm_id).values

print(
    f"groups: {len(groups):,} (in {groups_rows.tile_id.nunique():,} tiles, {groups_rows.tile_id.nunique() / len(tiles):.1%} of tiles)"
)
print(
    groups[["area_m2", "box_long", "fill", "crowns_equiv", "tiles"]]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.99])
    .round(2)
    .to_string()
)
total = tiles_m.area.sum()
print(
    f"\ngroup area: {groups.area_m2.sum() / 1e6:.2f} km² of {total / 1e6:.2f} km² tiled ({groups.area_m2.sum() / total:.1%})"
)
print(
    "five largest groups (m²):", groups.nlargest(5, "area_m2").set_index("osm_id").area_m2.round(0).to_dict()
)

inside = gpd.sjoin(trees[["geometry"]], groups[["osm_id", "geometry"]], predicate="within")
print(
    f"tree points inside a group: {inside.index.nunique():,}; groups holding >= 1 tree point: {inside.osm_id.nunique():,}"
)

tile_area = tiles_m.set_index("tile_id").area
cover = (
    (groups_rows.assign(a=groups_rows.area).groupby("tile_id").a.sum() / tile_area).fillna(0).clip(upper=1)
)
print(
    "\nshare of a tile covered by groups:",
    cover.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(3).to_dict(),
)
print(f"tiles > 80% group (dropped under the masking option's 80% rule): {(cover > 0.8).sum():,}")

groups: 10,638 (in 16,326 tiles, 33.7% of tiles)
         area_m2  box_long      fill  crowns_equiv     tiles
count   10638.00  10638.00  10638.00      10638.00  10638.00
mean      414.43     21.97      0.58         32.98      2.30
std      4716.51     24.59      0.11        375.33      4.37
min         0.00      0.00      0.02          0.00      1.00
10%        42.03      9.58      0.44          3.34      1.00
25%        74.96     12.75      0.51          5.97      1.00
50%       134.99     17.51      0.59         10.74      2.00
75%       251.14     25.06      0.66         19.99      3.00
90%       469.70     35.83      0.71         37.38      4.00
99%      2257.95     85.52      0.79        179.68      7.00
max    248992.49    882.27      0.88      19814.19    207.00

group area: 4.41 km² of 68.86 km² tiled (6.4%)
five largest groups (m²): {1534493829: 248992.0, 1533959071: 234121.0, 1533960603: 224530.0, 1528168813: 111254.0, 1533959838: 86204.0}
tree points inside a group: 390; gr

In [7]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
bins = np.logspace(-0.5, 3, 60)
ax[0].hist(d.clip(lower=0.4), bins=bins, alpha=0.6, density=True, label="tree: diameter_crown")
ax[0].hist(
    groups.box_long.clip(lower=0.4), bins=bins, alpha=0.6, density=True, label="tree_group: box long side"
)
for x, name in [
    (WHOLE_OBJECT_M, "window-stride"),
    (WINDOW_M, "window"),
    (ANCHOR_MIN_PX * WINDOW_M / MODEL_PX, "min anchor"),
]:
    ax[0].axvline(x, ls="--", c="k", lw=0.8)
    ax[0].text(x, ax[0].get_ylim()[1] * 0.9, name, rotation=90, ha="right", fontsize=8)
ax[0].set_xscale("log")
ax[0].set_xlabel("metres")
ax[0].legend(fontsize=8)
ax[0].set_title("box size: single trees vs groups (density)")
groups.fill.plot.hist(bins=40, ax=ax[1], title="group area / its box area (fill)")
groups.crowns_equiv.clip(upper=100).plot.hist(
    bins=50, ax=ax[2], title="group area in median-crown units (clipped 100)"
)
plt.tight_layout()
plt.show()

## 4. What each labelling option gives the detector

* **A – current setup:** points → boxes from `diameter_crown`; groups → ignore mask (as for OAM-TCD canopy groups).
* **B – one box per group, same class:** points → boxes; each group → one `tree` box.
* **C – two classes:** as B, but groups get a separate `tree_group` class (the model becomes 2-class; the release head is 1-class).

The table below counts boxes, their sizes relative to the serving window and anchors, and how often a group box would sit on top of point-tree boxes (conflicting targets).

In [8]:
fallback_d = float(trees.d.median())
trees["d_box"] = trees.d.fillna(fallback_d).clip(1, 30)
point_boxes = trees.geometry.buffer(trees.d_box / 2, cap_style="square")

long_g = groups.box_long
rows = {
    "boxes from points": len(trees),
    "boxes from groups": 0,
    "masked area share": groups.area_m2.sum() / total,
}
summary = pd.DataFrame(
    {
        "A: mask groups": rows,
        "B/C: box per group": {**rows, "boxes from groups": len(groups), "masked area share": 0.0},
    }
).T
summary["group boxes > window-stride (split at serving)"] = [0, (long_g > WHOLE_OBJECT_M).mean()]
summary["group boxes > window (never whole)"] = [0, (long_g > WINDOW_M).mean()]
summary["group boxes > largest anchor"] = [0, (long_g > ANCHOR_MAX_PX * WINDOW_M / MODEL_PX).mean()]
print(summary.T.to_string(float_format=lambda v: f"{v:,.3f}"))

hits = gpd.sjoin(
    gpd.GeoDataFrame(geometry=point_boxes.envelope),
    gpd.GeoDataFrame(geometry=groups.envelope),
    predicate="intersects",
)
print(
    f"\npoint-tree boxes overlapping a group's box (conflicting targets under B/C): {hits.index.nunique() / len(trees):.1%} of trees,"
    f" in {hits.index_right.nunique() / len(groups):.1%} of groups"
)
print(
    f"group pieces per group (clipped at tile edges): mean {groups.tiles.mean():.1f}, max {groups.tiles.max()}"
    " -> a crop cuts large groups into partial boxes; the 40% rule decides which survive"
)

                                                A: mask groups  B/C: box per group
boxes from points                                   95,992.000          95,992.000
boxes from groups                                        0.000          10,638.000
masked area share                                        0.064               0.000
group boxes > window-stride (split at serving)           0.000               0.399
group boxes > window (never whole)                       0.000               0.074
group boxes > largest anchor                             0.000               0.071

point-tree boxes overlapping a group's box (conflicting targets under B/C): 6.8% of trees, in 34.0% of groups
group pieces per group (clipped at tile edges): mean 2.3, max 207 -> a crop cuts large groups into partial boxes; the 40% rule decides which survive


## 5. Splits and leakage

The card's split: 10 contiguous regions per project, one to test, one to validation, with no buffer ("tiles at a region edge border tiles of another split; a feature crossing an edge has parts in both"). Our rule for HOT data is a block split plus a 1 km buffer.

In [9]:
print(
    tiles.groupby("split")
    .agg(
        tiles=("tile_id", "size"),
        labelled=("num_features", lambda s: (s > 0).mean()),
        labels=("num_features", "sum"),
    )
    .round(3)
)
cross = labels.groupby("osm_id").split.nunique()
print(f"\nOSM features appearing in more than one split: {(cross > 1).sum():,}")

centres = {s: np.c_[g.geometry.centroid.x, g.geometry.centroid.y] for s, g in tiles_m.groupby("split")}
train_tree = cKDTree(centres["train"])
for s in ("validation", "test"):
    dist, _ = train_tree.query(centres[s])
    print(
        f"{s}: tiles within 100 m / 500 m / 1 km of a train tile: "
        + " / ".join(f"{(dist < r).mean():.1%}" for r in (100, 500, 1000))
    )

fig, ax = plt.subplots(figsize=(7, 5))
colours = {"train": "#9ecae1", "validation": "#fd8d3c", "test": "#e6550d"}
for s, g in tiles.groupby("split"):
    g.plot(ax=ax, color=colours[s], label=s, linewidth=0)
groups.to_crs(tiles.crs).boundary.plot(ax=ax, color="darkgreen", linewidth=0.3)
ax.set_title("splits (tiles) with tree-group outlines")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in colours.values()], labels=list(colours))
plt.show()

            tiles  labelled  labels
split                              
test         5627     0.701   12230
train       38705     0.713   92815
validation   4158     0.895   15389

OSM features appearing in more than one split: 35
validation: tiles within 100 m / 500 m / 1 km of a train tile: 6.5% / 35.0% / 63.1%
test: tiles within 100 m / 500 m / 1 km of a train tile: 6.3% / 34.0% / 71.0%


## 6. Imagery with labels

Mosaics of k × k adjacent tiles (k = 4 gives 1024 px, about 150 m) from the image split. Points are drawn as their `diameter_crown` box (yellow), groups as polygons (cyan) with their bounding box (magenta, dashed) — the box option B/C would train on.

Look for: unmapped trees (completeness), shrubs mapped as trees, crown-size accuracy of `diameter_crown`, and how much non-tree a group box takes in.

In [10]:
import io

import pyarrow.parquet as pq
from PIL import Image

img_table = pq.read_table(image_path, columns=["tile_id", "image"]).to_pandas()
images = dict(zip(img_table.tile_id, (v["bytes"] for v in img_table.image), strict=False))
del img_table
split_tiles = tiles[tiles.tile_id.isin(images)].set_index("tile_id")
coords = {(r.tile_x, r.tile_y): t for t, r in split_tiles.iterrows()}
Z = int(tiles.tile_z.iloc[0])
R = 6378137.0
TILE_M = 2 * np.pi * R / 2**Z  # web-mercator metres per tile
labels_3857 = labels.to_crs("EPSG:3857")
print(f"{len(images):,} {IMAGE_SPLIT} tiles loaded")


def mosaic(x0, y0, k):
    """RGB array of k x k tiles from (x0, y0), black where a tile is missing, and its 3857 origin."""
    out = np.zeros((256 * k, 256 * k, 3), np.uint8)
    for dx in range(k):
        for dy in range(k):
            t = coords.get((x0 + dx, y0 + dy))
            if t is not None:
                out[dy * 256 : (dy + 1) * 256, dx * 256 : (dx + 1) * 256] = np.asarray(
                    Image.open(io.BytesIO(images[t])).convert("RGB")
                )
    return out, (-np.pi * R + x0 * TILE_M, np.pi * R - y0 * TILE_M)


def mosaic_labels(x0, y0, k):
    """Labels of the mosaic in pixel coordinates: tree points with d, group polygons (deduplicated)."""
    ids = [coords[(x0 + dx, y0 + dy)] for dx in range(k) for dy in range(k) if (x0 + dx, y0 + dy) in coords]
    sel = labels_3857[labels_3857.tile_id.isin(ids)]
    minx, maxy = -np.pi * R + x0 * TILE_M, np.pi * R - y0 * TILE_M
    px_per_m3857 = 256 / TILE_M
    lat = np.arctan(np.sinh(np.pi * (1 - 2 * (y0 + k / 2) / 2**Z)))  # latitude of the mosaic centre

    def to_px(x, y):
        return ((x - minx) * px_per_m3857, (maxy - y) * px_per_m3857)

    pts = sel[sel["class"] == "tree"]
    tx, ty = to_px(pts.geometry.x.values, pts.geometry.y.values)
    # metres on the ground -> mercator pixels: divide by cos(lat)
    dpx = (
        pts.tags.map(lambda t: parse_diameter(t.get("diameter_crown"))).fillna(fallback_d).values
        * px_per_m3857
        / np.cos(lat)
    )
    grp = sel[sel["class"] == "tree_group"].dissolve("osm_id")
    grp_px = grp.geometry.affine_transform(
        [px_per_m3857, 0, 0, -px_per_m3857, -minx * px_per_m3857, maxy * px_per_m3857]
    )
    return pd.DataFrame({"x": tx, "y": ty, "dpx": dpx}), grp_px


def show(ax, x0, y0, k, title="", preds=None):
    img, _ = mosaic(x0, y0, k)
    pts, grp = mosaic_labels(x0, y0, k)
    ax.imshow(img)
    for g in grp:
        for part in getattr(g, "geoms", [g]):
            ax.fill(*part.exterior.xy, fc="cyan", ec="cyan", alpha=0.25, lw=1)
        x1, y1, x2, y2 = g.bounds
        ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="magenta", ls="--", lw=1))
    for r in pts.itertuples():
        ax.add_patch(
            plt.Rectangle((r.x - r.dpx / 2, r.y - r.dpx / 2), r.dpx, r.dpx, fill=False, ec="yellow", lw=1)
        )
        ax.plot(r.x, r.y, "r.", ms=3)
    if preds is not None:
        for r in preds.itertuples():
            ax.add_patch(
                plt.Rectangle((r.xmin, r.ymin), r.xmax - r.xmin, r.ymax - r.ymin, fill=False, ec="red", lw=1)
            )
    ax.set_xlim(0, img.shape[1])
    ax.set_ylim(img.shape[0], 0)
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def greenness(tile_id):
    """Share of pixels with excess green 2g - r - b > 0.1 (chromatic coordinates): vegetation of any kind."""
    a = np.asarray(Image.open(io.BytesIO(images[tile_id])).convert("RGB")).astype(np.float32)
    s = a.sum(axis=2) + 1e-6
    r, g, b = (a[..., i] / s for i in range(3))
    return float((2 * g - r - b > 0.1).mean())


split_tiles["green"] = [greenness(t) for t in split_tiles.index]


def full_blocks(k):
    """Origins of k x k blocks fully inside the image split, with label stats per block."""
    stats = split_tiles.assign(groups=cover.reindex(split_tiles.index).fillna(0))
    out = []
    for x, y in coords:
        if x % k or y % k:  # non-overlapping grid
            continue
        ids = [coords.get((x + dx, y + dy)) for dx in range(k) for dy in range(k)]
        if None in ids:
            continue
        s = stats.loc[ids]
        out.append(
            (
                x,
                y,
                int(s.num_features.sum()),
                float(s.groups.mean()),
                int((s.num_features == 0).sum()),
                float(s.green.mean()),
            )
        )
    return pd.DataFrame(out, columns=["x", "y", "labels", "group_cover", "empty_tiles", "green"])


K = 4
blocks = full_blocks(K)
print(f"{len(blocks)} complete {K}x{K} blocks in {IMAGE_SPLIT}")

5,627 test tiles loaded
303 complete 4x4 blocks in test


In [11]:
order = {
    "random": blocks.sample(frac=1, random_state=SEED),
    "most group cover": blocks.sort_values("group_cover", ascending=False),
    "most labels": blocks.sort_values("labels", ascending=False),
    "unlabelled, greenest": blocks[blocks.labels == 0].sort_values("green", ascending=False),
}
picks, used = {}, set()
for name, ranked in order.items():  # two blocks per category, never the same block twice
    ranked = ranked[[(x, y) not in used for x, y in zip(ranked.x, ranked.y, strict=False)]]
    picks[name] = ranked.head(2)
    used |= set(zip(picks[name].x, picks[name].y, strict=False))
fig, axes = plt.subplots(4, 2, figsize=(14, 28))
for row, (name, sel) in zip(axes, picks.items(), strict=False):
    for ax, b in zip(row, sel.itertuples(), strict=False):
        show(
            ax,
            b.x,
            b.y,
            K,
            f"{name}: x={b.x} y={b.y}, {b.labels} labels, group cover {b.group_cover:.0%}, {b.empty_tiles} empty tiles",
        )
plt.tight_layout()
plt.show()

**Completeness.** Tiles without labels become negatives in training ("no tree here"). If some are unmapped tree cover, the model is taught to miss trees. Excess-green share is a crude vegetation measure (grass and shrubs count too), so read it as an upper bound and check the greenest label-free tiles by eye.

In [12]:
split_tiles["no_labels"] = split_tiles.num_features == 0
print(split_tiles.groupby("no_labels").green.describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(3))
empty_green = split_tiles[split_tiles.no_labels]
for th in (0.3, 0.5, 0.8):
    print(
        f"label-free tiles with > {th:.0%} green pixels: {(empty_green.green > th).sum():,} ({(empty_green.green > th).mean():.1%})"
    )

greenest = empty_green.nlargest(12, "green")
fig, axes = plt.subplots(2, 6, figsize=(18, 6.5))
for ax, (t, r) in zip(axes.flat, greenest.iterrows(), strict=False):
    ax.imshow(Image.open(io.BytesIO(images[t])))
    ax.set_title(f"{t}\ngreen {r.green:.0%}", fontsize=8)
    ax.axis("off")
plt.suptitle(f"greenest label-free {IMAGE_SPLIT} tiles")
plt.tight_layout()
plt.show()

            count   mean    std  min    25%    50%    75%    90%  max
no_labels                                                            
False      3947.0  0.263  0.229  0.0  0.102  0.194  0.339  0.569  1.0
True       1680.0  0.226  0.295  0.0  0.020  0.068  0.339  0.779  1.0
label-free tiles with > 30% green pixels: 449 (26.7%)
label-free tiles with > 50% green pixels: 307 (18.3%)
label-free tiles with > 80% green pixels: 156 (9.3%)


In [13]:
# look at any block: set the origin tile (x, y) and size
X0, Y0, KK = blocks.x.iloc[0], blocks.y.iloc[0], 6
fig, ax = plt.subplots(figsize=(12, 12))
show(ax, X0, Y0, KK, f"x={X0} y={Y0} k={KK}")
plt.show()

## 7. Optional: release DeepForest model zero-shot

Runs `weecology/deepforest-tree` (our start weights) on random complete blocks, at the native 0.15 m and upsampled ×1.5 to about the 0.10 m it was trained on. It answers two questions for the discussion:

* **Points:** what share of mapped tree points fall inside a predicted box (point-in-box recall), and how predicted box size compares with `diameter_crown`.
* **Groups:** does the release model already split a group into crowns (several boxes per group) or box it whole?

Predictions far from every label hint at unmapped trees; predictions on the greenest label-free blocks measure the same directly. Set `RUN_DEEPFOREST = True` to run (installs `deepforest`, about 2 minutes; on CPU each 4 × 4 block takes about 30 s at native scale and 50 s at ×1.5).

Scores below 0.3 are dropped. In a local check on 3 blocks, ×1.5 upsampling gave fewer and smaller boxes than native scale, so native looks like the fair zero-shot setting here.

In [19]:
RUN_DEEPFOREST = True
N_BLOCKS = 12

In [20]:
if RUN_DEEPFOREST:
    import subprocess
    import sys

    # if pip reports that the session must restart, restart it and run all cells again
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deepforest"], check=True)

In [21]:
if RUN_DEEPFOREST:
    from deepforest import main as df_main
    from shapely.geometry import Point, box

    model = df_main.deepforest()
    model.load_model("weecology/deepforest-tree")

    def predict(img, scale):
        if scale != 1:
            img = np.asarray(Image.fromarray(img).resize((int(img.shape[1] * scale),) * 2, Image.BILINEAR))
        p = model.predict_tile(image=img.astype(np.float32), patch_size=400, patch_overlap=0.25)
        if p is None or len(p) == 0:
            return pd.DataFrame(columns=["xmin", "ymin", "xmax", "ymax", "score"])
        p = p[["xmin", "ymin", "xmax", "ymax", "score"]].copy()
        p[["xmin", "ymin", "xmax", "ymax"]] /= scale
        return p[p.score >= 0.3]

    eval_blocks = blocks[blocks.labels > 0].sample(
        min(N_BLOCKS, (blocks.labels > 0).sum()), random_state=SEED
    )
    records, size_pairs, examples = [], [], []
    for b in eval_blocks.itertuples():
        img, _ = mosaic(b.x, b.y, K)
        pts, grp = mosaic_labels(b.x, b.y, K)
        for scale in (1.0, 1.5):
            p = predict(img, scale)
            pboxes = [box(*r) for r in p[["xmin", "ymin", "xmax", "ymax"]].values]
            centres = [bx.centroid for bx in pboxes]
            in_box = [
                any(bx.contains(Point(x, y)) for bx in pboxes) for x, y in zip(pts.x, pts.y, strict=False)
            ]
            per_group = [sum(g.contains(c) for c in centres) for g in grp]
            label_zone = [
                box(r.x - r.dpx, r.y - r.dpx, r.x + r.dpx, r.y + r.dpx) for r in pts.itertuples()
            ] + list(grp)
            unexplained = sum(not any(z.contains(c) for z in label_zone) for c in centres)
            records.append(
                {
                    "block": (b.x, b.y),
                    "scale": scale,
                    "trees": len(pts),
                    "groups": len(grp),
                    "preds": len(p),
                    "point_in_box": float(np.mean(in_box)) if len(pts) else np.nan,
                    "boxes_per_group": float(np.mean(per_group)) if len(grp) else np.nan,
                    "groups_with_>=2_boxes": float(np.mean([n >= 2 for n in per_group]))
                    if len(grp)
                    else np.nan,
                    "unexplained_preds": unexplained / max(len(p), 1),
                }
            )
            if scale == 1.0:
                examples.append((b.x, b.y, p))
                for r in pts.itertuples():
                    hit = [bx for bx in pboxes if bx.contains(Point(r.x, r.y))]
                    if len(hit) == 1:
                        side = np.sqrt(hit[0].area) / r.dpx  # predicted side / diameter_crown side
                        size_pairs.append(side)
    res = pd.DataFrame(records)
    print(
        res.groupby("scale")[
            ["point_in_box", "boxes_per_group", "groups_with_>=2_boxes", "unexplained_preds"]
        ]
        .mean()
        .round(3)
    )

    empty_blocks = blocks[blocks.labels == 0].nlargest(3, "green")
    empty_preds = [(b.x, b.y, predict(mosaic(b.x, b.y, K)[0], 1.0)) for b in empty_blocks.itertuples()]
    print(
        "\npredicted trees on the greenest label-free blocks (native scale):",
        [len(p) for _, _, p in empty_preds],
        f"(each block is {K * K} tiles, about {K * 256 * GSD:.0f} m square)",
    )
    print(
        f"\npredicted box side / diameter_crown (native, single-hit trees): median {np.median(size_pairs):.2f},"
        f" IQR {np.percentile(size_pairs, 25):.2f}-{np.percentile(size_pairs, 75):.2f}"
    )

config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 180MB/s]


model.safetensors: reconstructing file:   0%|          |  0.00B /  129MB            

model.safetensors: downloading bytes:           |  0.00B            

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


... (103 outputs left out)

Output()

485 predictions in overlapping windows, applying non-max suppression
250 predictions kept after non-max suppression
No image path provided, root_dir of the output results dataframe will be None, since either images were directly provided or there were multiple image paths


/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


155 predictions in overlapping windows, applying non-max suppression
77 predictions kept after non-max suppression
No image path provided, root_dir of the output results dataframe will be None, since either images were directly provided or there were multiple image paths

predicted trees on the greenest label-free blocks (native scale): [40, 37, 14] (each block is 16 tiles, about 151 m square)

predicted box side / diameter_crown (native, single-hit trees): median 1.35, IQR 1.03-1.83


In [22]:
if RUN_DEEPFOREST:
    panels = [(x, y, p, "native, labelled") for x, y, p in examples[:2]] + [
        (x, y, p, "native, label-free") for x, y, p in empty_preds[:2]
    ]
    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    for ax, (x, y, p, what) in zip(axes.flat, panels, strict=False):
        show(ax, x, y, K, f"x={x} y={y}: predictions (red, {what}) over labels", preds=p)
    plt.tight_layout()
    plt.show()

## 8. Summary and questions for HOT

In [23]:
facts = {
    "tiles / area": f"{len(tiles):,} z{Z} tiles, {total / 1e6:.1f} km², one project (Freetown), GSD {GSD:.3f} m",
    "trees (points)": f"{len(trees):,}; diameter_crown parseable for {trees.d.notna().mean():.0%}, median {trees.d.median():.1f} m",
    "palms": f"{pal.mean():.0%} of trees carry leaf_type=palm",
    "tree groups": f"{len(groups):,} polygons, {groups.area_m2.sum() / total:.1%} of the area; median {groups.area_m2.median():.0f} m² (≈{groups.crowns_equiv.median():.0f} median crowns)",
    "group box fill": f"median {groups.fill.median():.2f} (a group box is ~{1 - groups.fill.median():.0%} non-group)",
    "groups larger than window-stride / window": f"{(long_g > WHOLE_OBJECT_M).mean():.0%} / {(long_g > WINDOW_M).mean():.0%}",
    "crown overlap": f"{(trees.nn[both] < trees.d[both]).mean():.0%} of trees closer to a neighbour than their diameter",
    "split": "10 regions, no buffer; features cross split edges",
    f"label-free {IMAGE_SPLIT} tiles > 50% green": f"{(empty_green.green > 0.5).mean():.0%} of {len(empty_green):,}",
}
if RUN_DEEPFOREST:
    native = res[res.scale == 1.0].mean(numeric_only=True)
    facts["release model, native"] = (
        f"point-in-box {native.point_in_box:.0%}, {native.boxes_per_group:.1f} boxes per group,"
        f" {native.unexplained_preds:.0%} predictions away from any label"
    )
for key, value in facts.items():
    print(f"{key:>45}: {value}")

                                 tiles / area: 48,490 z20 tiles, 68.9 km², one project (Freetown), GSD 0.147 m
                               trees (points): 95,992; diameter_crown parseable for 96%, median 4.0 m
                                        palms: 20% of trees carry leaf_type=palm
                                  tree groups: 10,638 polygons, 6.4% of the area; median 135 m² (≈11 median crowns)
                               group box fill: median 0.59 (a group box is ~41% non-group)
    groups larger than window-stride / window: 40% / 7%
                                crown overlap: 11% of trees closer to a neighbour than their diameter
                                        split: 10 regions, no buffer; features cross split edges
            label-free test tiles > 50% green: 18% of 1,680
                        release model, native: point-in-box 57%, 4.6 boxes per group, 50% predictions away from any label


**Questions to settle with HOT**

1. **Role of this dataset.** It is far larger than the planned 150-chip adaptation set. Is it a base-training source, the adaptation set, the HOT benchmark region, or all three (with which split)?
2. **Tree groups – target.** One box per group (option B), a separate `tree_group` class (C), or ignore (A)? For B/C: how should groups larger than the serving window be handled (split into window-sized boxes, capped, or ignored above a size)? How is a group box evaluated (IoU against a box that is ~40% non-group)?
3. **What fAIr outputs.** OSM wants `natural=tree` points (+ `diameter_crown`) and `natural=tree_group` polygons. A box maps cleanly to a point + diameter; a group box is a poor polygon. Is a polygon (segmentation) for groups wanted later?
4. **Points → boxes.** Accept `diameter_crown` as the box size? Fallback for missing values, clamp range, and are repeated preset values (2.412, 2.231, …) trustworthy? Or train/evaluate tree points by point-in-box only?
5. **Completeness.** Are all trees in a validated task mapped? Label-free tiles become negatives; section 6 shows label-free tiles with dense tree cover. Should tiles with no labels be dropped, masked, or kept only where a validator confirmed "no trees"?
6. **Shrubs and palms.** Keep palms as trees (≈20% of points)? Drop d ≤ 1 m as shrubs?
7. **Split.** Use HOT's 10-region split or our block split with a 1 km buffer?
8. **Mixing with OAM-TCD.** If groups become boxes here, OAM-TCD canopy groups should get the same treatment for consistency (they are masked in our current setup), and the NEON/SelvaBox sources have no group class at all.